In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Import the class we just created
from Legacy_pipeline import LegacyPipeline 

# Initialize
legacy_pipe = LegacyPipeline()

In [ ]:
def visualize_legacy_steps(img_path, 
                                 use_adaptive=False, 
                                 adaptive_block_size=25, 
                                 adaptive_c=10,
                                 notch_band_width=125, 
                                 peak_safety_margin=19,
                                 filter_min_area=200,
                                 filter_max_area=1500,
                                 shape_epsilon_mult=0.04,
                                 box_padding=25):
    
    img = cv2.imread(img_path)
    if img is None:
        print("Image not found!")
        return
        
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 1. Thresholding
    if use_adaptive:
        mask = legacy_pipe._adaptive_thresholding(
            gray, 
            block_size=adaptive_block_size, 
            C=adaptive_c, 
        )
        thresh_title = f"Adaptive Threshold"
    else:
        hist = legacy_pipe._get_perimeter_histogram(gray, 15)
        peak_val = int(np.argmax(hist))
        thresh_val = legacy_pipe._find_peak_decay_threshold(hist, peak_val, safety_margin=peak_safety_margin)
        mask = legacy_pipe._thresholding(gray, thresh_val)
        thresh_title = f"Peak Decay Mask (Thresh: {thresh_val})"
        
    # 2. Hull & Concavity Extraction
    raw_notches, hull_mask = legacy_pipe._hull_concavity(mask, band_width=notch_band_width)
    
    # 3. Filtered Notches
    filtered_notches = legacy_pipe._filter_notches_by_geometry(
        raw_notches, min_area=filter_min_area, max_area=filter_max_area
    )
    
    # 4. Final Classification
    accepted_notches = legacy_pipe._classify_notch_shapes(
        filtered_notches, epsilon_mult=shape_epsilon_mult, box_padding=box_padding
    )
    
    # --- Plotting ---
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    axes[0].imshow(img_rgb); axes[0].set_title("Original Image")
    axes[1].imshow(mask, cmap='gray'); axes[1].set_title(thresh_title)
    axes[2].imshow(hull_mask, cmap='gray'); axes[2].set_title("Global Convex Hull")
    axes[3].imshow(raw_notches, cmap='gray'); axes[3].set_title("Raw Concavities")
    axes[4].imshow(filtered_notches, cmap='gray'); axes[4].set_title("Filtered Notches")
    
    axes[5].imshow(img_rgb)
    for n in accepted_notches:
        x1, y1, x2, y2 = n['coords']
        shape = n['shape']
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='cyan', facecolor='none')
        axes[5].add_patch(rect)
        axes[5].text(x1, y1-10, shape, color='yellow', weight='bold', fontsize=10, backgroundcolor='black')
    axes[5].set_title(f"Final Output ({len(accepted_notches)} notches)")
    
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Test it out on a specific image!
test_image_path = "" #FIXME: change this to a valid image path

# Toggle 'use_adaptive' to see the difference between the two methods
visualize_legacy_steps(test_image_path, use_adaptive=True, notch_band_width=150)
visualize_legacy_steps(test_image_path, use_adaptive=False, notch_band_width=150)